Setup:

In [1]:
import gc
gc.collect()

28

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch.cuda.empty_cache()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', use_fast=True)
model = AutoModelForCausalLM.from_pretrained('meta-llama/Meta-Llama-3-8B', dtype=torch.bfloat16, device_map="auto")
model.seqlen = model.config.max_position_embeddings

/home/mrajnoha/double-block-sparse/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 291/291 [00:11<00:00, 25.47it/s] 


In [4]:
from llm import find_layers

find_layers(model)

# length = 0
# for name, param in model.named_parameters():
#     if 'proj' in name:
#         print(name, param.shape)
#         length += 1
# print(f'Matrix count: {length}')

ImportError: attempted relative import with no known parent package

In [ ]:
model.hf_device_map

In [ ]:
q_proj_weight = model.model.layers[3].self_attn.q_proj.weight
print(q_proj_weight.shape)   # torch.Size([4096, 4096])
print(q_proj_weight.device)
print(q_proj_weight.dtype)
print(q_proj_weight)


k_proj_weight = model.model.layers[5].self_attn.k_proj.weight
print(k_proj_weight.shape)   # torch.Size([4096, 4096])
print(k_proj_weight.device)
print(k_proj_weight.dtype)
print(k_proj_weight)

Tests:

In [ ]:
from test_dbsf import *

In [ ]:
a = test_double_sparse(matrix=q_proj_weight, total_sp=0.5, b_bias=0.5, mid_dim_scale=1)

In [ ]:
b = test_hybrid(matrix=q_proj_weight, asp=0.25, bsp=0.25)

In [ ]:
c = test_2to4_A_B(matrix=q_proj_weight, sp=0.25, mid_dim_scale=1)

In [ ]:
d = test_1x2_2x1(matrix=q_proj_weight, sp=0.25, mid_dim_scale=1)

In [ ]:
e = test_mag_prune(matrix=q_proj_weight, sp=0.5)

In [ ]:
f = test_double_block_sparse(matrix=q_proj_weight, total_sp=0.5, b_bias=0.5, mid_dim_scale=2)

In [ ]:
g = test_double_block_sparse(matrix=q_proj_weight, total_sp=0.5, b_bias=0.5, mid_dim_scale=1)

In [ ]:
h = test_2to4_A_B(matrix=q_proj_weight, sp=0.5, mid_dim_scale=0.5)

Output:

In [ ]:
import seaborn as sns
sns.reset_orig()

from matplotlib import rcParams
rcParams['figure.figsize'] = 12.5, 6

outputs = {'dsf': a,
           'hybrid': b, 
           '2:4': c,
           '1x2': d,
           'mag_prune': e,
           'dbsf_2x_mid': f,
           'dbsf': g,
           '2:4_compr': h,
           }
alpha = [1, 0.5, 0.5, 1, 1, 1, 1, 1]

bars = sns.barplot(x = outputs.keys(), y = outputs.values())
bars.tick_params(axis='x', labelsize=12)
for bar, a in zip(bars.containers[0], alpha):
    bar.set_alpha(a)

Cleanup:

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()